# ⚡ Módulo 11 - Notebook 02: Arquitectura Spark y DataFrames

## 🏗️ Particiones, esquemas y tipos de datos distribuidos

**Libro:** Saliendo de lo Pandito  
**Módulo:** 11 - PySpark Core SparkSession  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** particiones y paralelismo  
✅ **Definir** esquemas explícitos con StructType  
✅ **Manejar** tipos de datos de Spark  
✅ **Optimizar** número de particiones  
✅ **Crear** DataFrames con esquema estricto

---

## 📋 Pre-requisitos

* ✅ Notebook 11_01 completado (Introducción a Spark)
* ✅ Conocimiento de SparkSession
* ✅ Familiaridad con tipos de datos

---

## 📚 Contenido

1. Particiones y Paralelismo
2. StructType y StructField
3. Tipos de Datos en Spark
4. Esquemas Explícitos vs Inferidos
5. Optimización de Particiones
6. Caso Integrador: DataFrame Empresarial con Esquema

---

## 💡 Por qué importa

**Esquemas y particiones son fundamentales:**

* 🏗️ **Esquemas:** Definen estructura y tipos
* ⚡ **Particiones:** Control del paralelismo
* 🎯 **Performance:** Optimización de ejecución
* 🛡️ **Calidad:** Validación de datos

**El corazón del procesamiento distribuido**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df_spark = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df_spark.count():,}")
    print(f"   🏛️ Particiones: {df_spark.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 Esquema del DataFrame:")
    df_spark.printSchema()
    
    print(f"\n🔍 Información de particiones:")
    print(f"   • Cada partición se procesa en paralelo")
    print(f"   • Spark divide los datos automáticamente")
    print(f"   • Número óptimo depende del cluster")
    
    print(f"\n🎯 Este notebook explorará:")
    print(f"   • Esquemas (StructType)")
    print(f"   • Tipos de datos de Spark")
    print(f"   • Optimización de particiones")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_spark = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Arquitectura Interna: Particiones y Esquemas

### 🗂️ Particiones: Divide y Vencerás

**Partición:** Un pedazo del DataFrame distribuido en un Executor.

**Concepto:**
```
DataFrame con 1000 filas, 4 particiones:

Partición 1 (Executor 1): Filas 1-250
Partición 2 (Executor 2): Filas 251-500
Partición 3 (Executor 3): Filas 501-750
Partición 4 (Executor 4): Filas 751-1000
```

**Ventaja:** Las 4 particiones se procesan **en paralelo**.

---

### ⚡ Paralelismo y Particiones

**Regla de oro:**
```
Número de particiones ≈ 2-3 × número de cores
```

**Ejemplo:**
* Cluster con 8 cores → 16-24 particiones óptimas
* Cluster con 32 cores → 64-96 particiones óptimas

**Ver particiones:**
```python
df.rdd.getNumPartitions()  # Consultar
```

**Cambiar particiones:**
```python
df_repartitioned = df.repartition(16)  # Más particiones
df_coalesced = df.coalesce(4)  # Menos particiones (sin shuffle)
```

---

### 📐 StructType: Esquemas Explícitos

**StructType** define la estructura de un DataFrame.

**Sintaxis:**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("nombre", StringType(), nullable=False),
    StructField("edad", IntegerType(), nullable=True),
    StructField("ciudad", StringType(), nullable=True)
])

data = [("Juan", 30, "Mendoza"), ("María", 25, "Buenos Aires")]
df = spark.createDataFrame(data, schema=schema)
```

**Ventajas:**
* ✅ **Performance:** No necesita inferir tipos
* ✅ **Validación:** Rechaza datos inválidos
* ✅ **Documentación:** Esquema explícito

---

### 🔤 Tipos de Datos en Spark

**Tipos principales:**

| Spark Type | Python | SQL | Ejemplo |
|------------|--------|-----|----------|
| **StringType** | str | STRING | "Hola" |
| **IntegerType** | int | INT | 42 |
| **LongType** | int | BIGINT | 9999999999 |
| **FloatType** | float | FLOAT | 3.14 |
| **DoubleType** | float | DOUBLE | 3.141592 |
| **BooleanType** | bool | BOOLEAN | True |
| **DateType** | date | DATE | 2024-01-01 |
| **TimestampType** | datetime | TIMESTAMP | 2024-01-01 10:30:00 |
| **DecimalType** | Decimal | DECIMAL(10,2) | 1234.56 |

**Importar:**
```python
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    DateType, TimestampType
)
```

---

### 🔍 Esquema Inferido vs Explícito

**Inferido (automático):**
```python
df = spark.read.csv("file.csv", header=True, inferSchema=True)
# Spark adivina los tipos
```

**Explícito (recomendado en producción):**
```python
schema = StructType([
    StructField("fecha", DateType()),
    StructField("ventas", DoubleType())
])
df = spark.read.csv("file.csv", header=True, schema=schema)
# Tipos garantizados
```

**Comparación:**

| Aspecto | Inferido | Explícito |
|---------|----------|----------|
| **Velocidad** | Lento (escanea datos) | Rápido |
| **Precisión** | Puede fallar | 100% preciso |
| **Validación** | No | Sí |
| **Uso** | Exploración | Producción |

---

### 🔧 Operaciones con Particiones

**1️⃣ Repartition (shuffle completo):**
```python
# Redistribuir datos (costoso)
df_new = df.repartition(16)
```

**2️⃣ Coalesce (reducir sin shuffle):**
```python
# Reducir particiones (barato)
df_new = df.coalesce(4)
```

**3️⃣ Repartition por columna:**
```python
# Agrupar por columna (útil para joins)
df_new = df.repartition("sucursal_id")
```

**Regla:**
* Aumentar particiones → `repartition()`
* Reducir particiones → `coalesce()`

---

### 💼 Caso de Uso: DataFrame Empresarial

**Problema:** Cargar ventas con esquema estricto.

```python
from pyspark.sql.types import *

# Definir esquema
schema = StructType([
    StructField("fecha", DateType(), nullable=False),
    StructField("sucursal_id", IntegerType(), nullable=False),
    StructField("producto", StringType(), nullable=False),
    StructField("ventas", DecimalType(10, 2), nullable=False),
    StructField("costo", DecimalType(10, 2), nullable=True)
])

# Cargar con esquema
df = spark.read.csv("/path/ventas.csv", schema=schema, header=True)

# Validación automática: rechaza filas inválidas
```

**Ventaja:** Errores detectados en carga, no en análisis.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType,
    DateType, TimestampType, BooleanType, DecimalType
)
from pyspark.sql.functions import col
import warnings
warnings.filterwarnings('ignore')

print("🏗️ ARQUITECTURA SPARK: Particiones y Esquemas")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Particiones y paralelismo")
print("  • StructType y StructField")
print("  • Tipos de datos de Spark")
print("  • Esquemas explícitos vs inferidos")

print("\n📖 Métodos clave:")
print("  - df.rdd.getNumPartitions()  # Ver particiones")
print("  - df.repartition(n)  # Cambiar particiones")
print("  - StructType([StructField(...)])  # Definir esquema")
print("  - df.printSchema()  # Ver esquema")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🏗️ Particiones y esquemas con datos reales de Los Andes Market

### 📐 Esquema real de `ventas_mensuales_mendoza_h3`

La tabla tiene columnas con tipos Spark específicos:

```python
df_spark.printSchema()
# root
#  |-- sucursal_id: string
#  |-- sucursal_nombre: string
#  |-- zona: string
#  |-- lat: double
#  |-- lon: double
#  |-- ventas: double
#  |-- fecha: date
#  |-- h3_index: string
#  |-- h3_res8: string
#  |-- h3_res7: string
```

---

### 🗂️ Particiones en Databricks

En Databricks Free Edition, las tablas Delta se particionan automáticamente. Podemos:

```python
# Ver particiones actuales
df.rdd.getNumPartitions()

# Reparticionar (con shuffle — costoso)
df_repart = df.repartition(8)

# Coalesce (sin shuffle — barato)
df_coal = df.coalesce(2)
```

---

### 💡 Preguntas de negocio
* ¿Cuántas particiones tiene la tabla de ventas?
* ¿Cambia el rendimiento al reparticionar?
* ¿Qué tipo de dato es cada columna? (StructType explícito)

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, DateType, IntegerType
)
from pyspark.sql.functions import col, sum as _sum, count, avg, desc
import time

print("🏗️ PARTICIONES Y ESQUEMAS CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_spark is not None:
    print("\n1️⃣  ESQUEMA INFERIDO vs EXPLÍCITO")
    print("-"*70)

    print("\n   Esquema inferido (de la tabla Delta):")
    df_spark.printSchema()

    # Definir esquema explícito para Los Andes Market
    schema_los_andes = StructType([
        StructField("sucursal_id", StringType(), nullable=False),
        StructField("sucursal_nombre", StringType(), nullable=False),
        StructField("zona", StringType(), nullable=True),
        StructField("lat", DoubleType(), nullable=True),
        StructField("lon", DoubleType(), nullable=True),
        StructField("ventas", DoubleType(), nullable=False),
        StructField("fecha", DateType(), nullable=False),
        StructField("h3_index", StringType(), nullable=True),
        StructField("h3_res8", StringType(), nullable=True),
        StructField("h3_res7", StringType(), nullable=True),
    ])
    print(f"\n   Esquema explícito definido con StructType:")
    print(f"   {len(schema_los_andes.fields)} campos definidos")
    print("\n   💡 nullable=False en sucursal_id y ventas (no pueden ser nulos)")

    print("\n" + "="*70)
    print("\n2️⃣  PARTICIONES: Explorar y modificar")
    print("-"*70)

    n_part_original = df_spark.rdd.getNumPartitions()
    print(f"\n   Particiones originales: {n_part_original}")

    # Reparticionar a 8
    start = time.time()
    df_repart = df_spark.repartition(8)
    df_repart.count()  # acción para materializar
    t_repart = time.time() - start
    print(f"\n   repartition(8): {df_repart.rdd.getNumPartitions()} particiones")
    print(f"   ⏱️  Tiempo (con shuffle): {t_repart:.2f}s")

    # Coalesce a 2
    start = time.time()
    df_coal = df_repart.coalesce(2)
    df_coal.count()
    t_coal = time.time() - start
    print(f"\n   coalesce(2): {df_coal.rdd.getNumPartitions()} particiones")
    print(f"   ⏱️  Tiempo (sin shuffle): {t_coal:.2f}s")
    print(f"\n   💡 coalesce es más rápido porque no hace shuffle")

    print("\n" + "="*70)
    print("\n3️⃣  TIPOS DE DATOS: Verificar y convertir")
    print("-"*70)

    # Verificar tipos
    tipos = {f.name: str(f.dataType) for f in df_spark.schema.fields}
    print("\n   Tipos de datos del DataFrame:")
    for nombre, tipo in tipos.items():
        print(f"      {nombre}: {tipo}")

    # Convertir ventas a IntegerType (de DoubleType)
    df_int = df_spark.withColumn("ventas_int", col("ventas").cast(IntegerType()))
    print("\n   Nueva columna 'ventas_int' (cast IntegerType):")
    df_int.select("ventas", "ventas_int").show(5)

    print("\n" + "="*70)
    print("\n4️⃣  PARTICIONAR POR COLUMNA: zona")
    print("-"*70)

    df_part_zona = df_spark.repartition("zona")
    print(f"\n   repartition('zona'): {df_part_zona.rdd.getNumPartitions()} particiones")
    print("   💡 Particionar por columna agrupa datos de la misma zona juntos")
    print("   Útil para joins y agregaciones frecuentes por zona")

    print("\n" + "="*70)
    print("\n5️⃣  ESQUEMA ESTRICTO: Validar datos")
    print("-"*70)

    # Crear DataFrame con esquema explícito desde los datos
    df_strict = spark.createDataFrame(df_spark.rdd, schema=schema_los_andes)
    print(f"\n   DataFrame creado con esquema explícito: {df_strict.count():,} filas")
    print("   ✅ Valida que los tipos coinciden")

    # Verificar nulos
    from pyspark.sql.functions import isnan, when
    nulos = df_spark.select([count(when(col(c).isNull(), c)).alias(c) for c in ["sucursal_id", "ventas", "fecha"]])
    print("\n   Conteo de nulos en campos críticos:")
    nulos.show()
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del notebook 11_02

### ✅ Lo que aprendiste

1. **Particiones y paralelismo:**
   - Cada partición se procesa en un executor diferente (en paralelo)
   - `df.rdd.getNumPartitions()` consulta el número actual
   - Regla: particiones ≈ 2-3 × número de cores del cluster

2. **StructType y StructField:**
   - `StructType([StructField('nombre', StringType(), nullable=False)])`
   - Define esquema explícito con tipos garantizados
   - `nullable=False` rechaza nulos en esa columna

3. **Tipos de datos de Spark:**
   - `StringType`, `IntegerType`, `DoubleType`, `DateType`, `TimestampType`
   - `DecimalType(precision, scale)` para valores monetarios
   - Mapeo: Python `str` → `StringType`, `int` → `IntegerType`, `float` → `DoubleType`

4. **Esquema explícito vs inferido:**
   - Inferido: Spark escanea datos para adivinar tipos (lento, puede fallar)
   - Explícito: tipos garantizados, rápido, valida en carga
   - Producción: SIEMPRE esquema explícito

5. **Optimización de particiones:**
   - `repartition(n)` — aumenta particiones (con shuffle, costoso)
   - `coalesce(n)` — reduce particiones (sin shuffle, barato)
   - `repartition('columna')` — particiona por valor de columna (útil para joins)

---

### 🎯 Reglas de Oro

👉 **Regla #1: En producción, SIEMPRE esquema explícito**
```python
# MALO: inferSchema escanea todos los datos y puede equivocarse
df = spark.read.csv('ventas.csv', header=True, inferSchema=True)

# BUENO: esquema explícito, tipos garantizados
schema = StructType([
    StructField('fecha', DateType(), nullable=False),
    StructField('ventas', DecimalType(10, 2), nullable=False)
])
df = spark.read.csv('ventas.csv', header=True, schema=schema)
```

👉 **Regla #2: coalesce para reducir, repartition para aumentar**
```python
# MALO: repartition(4) para reducir (genera shuffle innecesario)
df_small = df.repartition(4)

# BUENO: coalesce para reducir (sin shuffle)
df_small = df.coalesce(4)
# repartition solo para aumentar o repartir por columna
df_big = df.repartition(32)
df_by_sucursal = df.repartition('sucursal_id')
```

👉 **Regla #3: Particiones ≈ 2-3 × cores del cluster**
```python
# MALO: 200 particiones en un cluster de 4 cores
# La mayoría de los executors esperan, overhead de shuffle
df = df.repartition(200)

# BUENO: 8-12 particiones para 4 cores
df = df.repartition(10)  # 4 cores × 2.5 = 10
# Cada executor procesa 2-3 particiones secuencialmente
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Ver número de particiones | `df.rdd.getNumPartitions()` |
| Aumentar particiones | `df.repartition(n)` (con shuffle) |
| Reducir particiones | `df.coalesce(n)` (sin shuffle) |
| Particionar por columna | `df.repartition('col')` |
| Definir esquema | `StructType([StructField(...)])` |
| Ver esquema | `df.printSchema()` |
| Cargar CSV con esquema | `spark.read.csv(path, schema=schema)` |
| Dinero/decimales | `DecimalType(10, 2)` |
| Fechas | `DateType()` o `TimestampType()` |
| Exploración (EDA) | `inferSchema=True` (aceptable) |
| Producción / ETL | Esquema explícito (obligatorio) |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🏗️ ¡Arquitectura Spark y DataFrames dominados!</h3>
  <p><i>"Esquemas explícitos y particiones óptimas: los dos pilares de la performance en Spark."</i></p>
</div>